# Step 4: AWQ — 显著通道权重缩放 + per-group INT4

**目标**：实现 AWQ（Lin 2023, *Activation-aware Weight Quantization*）的核心思想——**weight-only W4A16**：用 per-group 对称 INT4 量化权重、激活保持 FP16。关键技巧是"显著通道缩放"——对激活幅值大的 channel，把权重预先放大再量化，降低该 channel 的相对量化误差。

**对应 OUTLINE 课时**：1.4 AWQ（~50 分钟）。

> AWQ 与 SmoothQuant（s3）都用"激活幅值指导缩放"，但方向相反：SmoothQuant 把激活难度**搬进权重**（激活也量化，W8A8）；AWQ **只量化权重**（W4A16，激活永远 FP16），缩放纯粹是为了让权重在"重要 channel"上量化得更准。

In [ ]:
%%capture
import math, json, pathlib
import torch
import torch.nn as nn
import ipytest
ipytest.autoconfig()

In [ ]:
# Setup cell：notebook 向上发现模块根（含 steps/ + pyproject.toml），绝不依赖裸相对路径。
# 规范见 course/NOTEBOOK_CONVENTIONS.md 第 2 节。所有文件路径从 MODULE_ROOT 派生。
def _find_module_root(start):
    p = pathlib.Path(start).resolve()
    for cand in [p, *p.parents]:
        if (cand / "steps").is_dir() and (cand / "pyproject.toml").exists():
            return cand
    raise RuntimeError("找不到模块根（含 steps/ + pyproject.toml）；请在模块目录内 cd course/m1-activation-outliers 启动 jupyter")

MODULE_ROOT    = _find_module_root(pathlib.Path.cwd())
MODEL_DIR      = MODULE_ROOT / "models" / "Qwen2.5-7B-Instruct"      # 与 scripts/download_model.sh 一致
TINY_MODEL_DIR = MODULE_ROOT / "models" / "Qwen2.5-0.5B-Instruct"     # L3 先在 0.5B 上验，再上 7B
OUT_ROOT       = MODULE_ROOT / "out"                                 # 已 gitignore
OUT_ROOT.mkdir(parents=True, exist_ok=True)
print("MODULE_ROOT:", MODULE_ROOT)
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability()
    print("GPU:", torch.cuda.get_device_name(), "（sm{}{}，cap={}）".format(cap[0], cap[1], cap),
          "— 支持 FP8" if cap >= (8, 9) else "")
else:
    print("无 GPU（仅 L1/L2 可跑；L3 自动跳过）")

## 原理

### Weight-only：为什么 W4A16 有意义？

推理时 LLM 是**访存瓶颈**（memory-bound）——算力富余，瓶颈是搬权重的带宽。INT4 权重 = FP16 的 1/4 磁盘/带宽，吞吐近 4×。激活保持 FP16（计算时即时反量化），无需校准激活。所以 W4A16 是"用更窄的权换取带宽"，精度损失主要来自权重量化。

### per-group 对称 INT4 量化

把权重按 `group_size`（如 128）切组，每组独立算 scale（对称 absmax）：
$$s_g = \frac{7}{\max|W_g|}, \quad W_q[g] = \text{round}(W_g \cdot s_g) \in [-7, 7]$$
反量化 $\hat{W}_g = W_q[g] / s_g$。per-group 比 per-channel 更细（每 128 个权重一个 scale），INT4 也能保精度。对称 = 量化范围 $[-7,7]$（INT4 符号位剩 7 个量级）。

### 显著通道缩放（AWQ 的精髓）

直接量化会"一视同仁"。但激活告诉我们：某些 channel 的激活幅值大（=s1 的 outlier，但 AWQ 看的是"显著性"=激活幅值）。对**显著 channel**（激活幅值大的列 $j$），其权重的量化误差对输出的影响被大激活**放大**。AWQ 的解法：

预先给显著 channel 的权重乘一个缩放 $s_j$（$s_j$ 由激活幅值决定），量化时这些权重的相对值变大 → 量化相对误差变小：
$$\tilde{W}[:,j] = s_j \cdot W[:,j], \quad \text{quantize}(\tilde{W}), \quad \text{推理时激活相应除 } s_j \text{ 抵消}$$
即 $Y = X W = (X / s) \cdot (s \cdot W) = \tilde{X} \tilde{W}$——同 SmoothQuant 的等价套路，但**只为了量化精度**（激活不量化，所以 $\tilde{X}$ 也保持 FP16，无需校准）。

**关键差异 vs SmoothQuant**：SmoothQuant 的 $s$ 平衡激活/权重两侧（因为两者都量化）；AWQ 的 $s$ **只优化权重量化误差**（激活不量化，所以 $s$ 取让"显著 channel 权重相对误差最小"的值——论文用 $s_j = \max|X_j|^{\alpha}$ 经验取 $\alpha \approx 0.5$ 的网格搜索最优）。本节我们用简化版：$s_j$ 正比于激活 channel 的平均幅值。

## 本步填空

1. **`awq_weight_scale(activations, alpha=0.5)`** —— 由激活幅值算每 channel 的缩放 $s_j$。
2. **`per_group_symmetric_quant(w, group_size, n_bits)`** —— per-group 对称 INT 量化，返回 `(量化整数, scale)`。
3. （判断）AWQ 预缩放后，**显著 channel** 的量化误差下降（整体均值可能反而升——见 L2 原因）。

In [ ]:
def awq_weight_scale(activations, alpha=0.5):
    """由激活幅值算每 channel（W 的列）的 AWQ 缩放 s_j。

    参数
    ----
    activations : torch.Tensor [..., c]，最后一维 c = channel = W 的列数 n（注意 s4 约定 W 形状为 [k, n]，n 是列 channel）。
    alpha : float in (0,1]，经验值（论文网格搜索 ~0.5）。越大越强调显著 channel。

    返回
    ----
    torch.Tensor [k]，每 channel 的缩放 s_j，float32，全正。
      简化定义（对齐 AWQ 直觉）：s_j = (mean|X[:,j]|)^alpha，再做归一化让 s 的均值=1
      （归一化保证整体量化范围不被系统性偏移）。

    提示
    ----
      - 先 flatten 到 [N, k]，取每 channel 绝对值均值 → [k]。
      - s = (mean_abs ** alpha)。
      - 归一化：s = s / s.mean()（让平均 scale=1，避免整体放大/缩小）。
      - 防 0：mean_abs 先 clamp_min(1e-8)。
    """
    # TODO: 返回 [k] 的 AWQ 缩放向量。
    raise NotImplementedError


def per_group_symmetric_quant(w, group_size=128, n_bits=4):
    """per-group 对称 INT 量化（weight-only，W4 默认）。

    参数
    ----
    w : torch.Tensor [k, n] 权重（或 [out, in]，按最后一维分组皆可；这里按 **flatten 后的行** 分组）。
        约定：把 w 当成 2D，沿 dim=1（每行）按 group_size 切组量化。
    group_size : int，每组元素数（默认 128）。
    n_bits : int，位宽（默认 4 = INT4）。量化范围 qmax = 2^(n_bits-1) - 1（如 INT4 → 7）。

    返回
    ----
    (w_q, scale) 元组：
      - w_q : 与 w 同形，dtype 由 n_bits 决定（INT4 无原生 dtype，用 int8 容器存），范围 [-qmax, qmax]。
      - scale : torch.Tensor，形状 [k, n//group_size]，每组一个反量化 scale。

    per-group 算法（每组独立 absmax 对称量化）：
      对每行每段 g：max_g = max(|w[row, g]|); s_g = qmax / max_g; w_q = round(w * s_g).clamp(-qmax,qmax)
      反量化：w_hat = w_q / s_g。

    提示
    ----
      - qmax = 2**(n_bits-1) - 1。
      - 需要把每行 reshape 成 [n // group_size, group_size] 处理；若 n 不整除 group_size，
        可以先 pad 或只处理整除部分（本课假设整除）。
      - 防 0 除：max_g.clamp_min(1e-8)。
      - 一种实现：w 按 [k, n//gs, gs] reshape，对最后一维取 amax，广播算 scale 与 w_q。
      - 返回 w_q 存 int8 容器（torch 无 int4），scale 存 float32。
    """
    # TODO: 实现 per-group 对称量化。
    raise NotImplementedError


# 脚手架（提供）：用 scale 预缩放 W 再量化，比较 scaled vs unscaled 的量化误差。
def quant_error(original, dequantized):
    """相对量化误差：mean|orig - deq| / mean|orig|。"""
    return ((original - dequantized).abs().mean() / original.abs().mean().clamp_min(1e-12)).item()

In [ ]:
%%ipytest -qq

def test_awq_weight_scale_shape_and_positive():
    a = torch.randn(8, 16).abs() + 0.1
    s = awq_weight_scale(a, alpha=0.5)
    assert s.shape == torch.Size([16])
    assert (s > 0).all()
    # 归一化：均值应接近 1
    assert abs(s.mean().item() - 1.0) < 1e-4

def test_awq_weight_scale_emphasizes_significant_channels():
    # channel 3 激活幅值远大
    a = torch.ones(4, 8); a[:, 3] *= 100.0
    s = awq_weight_scale(a, alpha=0.5)
    assert s[3].item() > s[0].item() * 3, "显著 channel 缩放应更大"

def test_per_group_symmetric_quant_shape_and_dtype():
    w = torch.randn(8, 256)
    wq, sc = per_group_symmetric_quant(w, group_size=128, n_bits=4)
    assert wq.shape == w.shape
    assert sc.shape == torch.Size([8, 2])   # 256/128 = 2 组/行
    assert wq.dtype == torch.int8
    assert wq.abs().max().item() <= 7        # INT4 范围 [-7,7]

def test_per_group_symmetric_quant_roundtrip_low_err():
    torch.manual_seed(0)
    w = torch.randn(16, 256) * 0.1
    wq, sc = per_group_symmetric_quant(w, group_size=128, n_bits=4)
    # 反量化
    k, n = w.shape; gs = 128
    deq = (wq.float().reshape(k, n//gs, gs) / sc.unsqueeze(-1)).reshape(k, n)
    err = quant_error(w, deq)
    assert err < 0.15, f"INT4 per-group 反量化误差应 <15%（随机权重 INT4 粗糙），实际 {err:.4f}"

def test_scaled_quant_lower_error_than_unscaled():
    """判断型：AWQ 预缩放显著 channel 后，权重量化误差应低于不缩放。"""
    torch.manual_seed(2)
    k, n = 8, 256
    w = torch.randn(k, n) * 0.1
    # 模拟激活：少数 channel 显著
    a = torch.ones(50, n); a[:, 30] *= 50.0; a[:, 100] *= 40.0
    s = awq_weight_scale(a, alpha=0.5)
    # 不缩放直接量化
    wq1, sc1 = per_group_symmetric_quant(w, group_size=128, n_bits=4)
    deq1 = (wq1.float().reshape(k, n//128, 128) / sc1.unsqueeze(-1)).reshape(k, n)
    err_unscaled = quant_error(w, deq1)
    # 缩放后量化（W 乘 s，量化，反量化再除 s）
    ws = w * s.unsqueeze(0)
    wq2, sc2 = per_group_symmetric_quant(ws, group_size=128, n_bits=4)
    deq2_scaled = (wq2.float().reshape(k, n//128, 128) / sc2.unsqueeze(-1)).reshape(k, n)
    deq2 = deq2_scaled / s.unsqueeze(0)
    err_scaled = quant_error(w, deq2)
    # AWQ 的目标：显著 channel 误差降低（整体误差通常也降，但关键是显著 channel）
    sig_err_unscaled = (w[:, 30] - deq1[:, 30]).abs().mean() / w[:, 30].abs().mean().clamp_min(1e-12)
    sig_err_scaled = (w[:, 30] - deq2[:, 30]).abs().mean() / w[:, 30].abs().mean().clamp_min(1e-12)
    assert sig_err_scaled.item() < sig_err_unscaled.item(), \
        f"显著 channel 量化误差应下降 {sig_err_scaled:.4f} < {sig_err_unscaled:.4f}"

## L2：tiny 验证（CPU）—— AWQ 降低显著 channel 的量化误差

合成带显著 channel 的激活 + tiny Qwen2 权重，验证 AWQ 缩放的核心效果：**显著 channel 的量化误差下降**。

> **预期一个反直觉现象（点破它，免得你怀疑实现有 bug）**：AWQ-scaled 的**整体**量化误差（对所有 channel 取均值）可能**高于** unscaled——甚至翻倍。这不是错误，正是 AWQ 的机制：它把缩放预算集中在显著 channel（激活幅值大、对最终输出贡献大的列），代价是非显著 channel 的量化误差上升。整体均值被大量非显著 channel 拉高，但**真实推理的损失由「激活 × 权重」的加权和决定**——显著 channel 误差降下来，对 perplexity 的影响远大于整体均值的变化。所以评判 AWQ 要看「显著 channel 误差」而非「整体均值」，下面的打印会逐 channel 把这点讲清楚。

In [ ]:
# 合成验证
torch.manual_seed(5)
k, n = 16, 256
w = torch.randn(k, n) * 0.08
a = torch.ones(64, n); a[:, 30] *= 60.0; a[:, 150] *= 45.0   # 显著 channel
s = awq_weight_scale(a, alpha=0.5)

# unscaled INT4
wq1, sc1 = per_group_symmetric_quant(w, 128, 4)
deq1 = (wq1.float().reshape(k, 2, 128) / sc1.unsqueeze(-1)).reshape(k, n)
# scaled INT4
ws = w * s.unsqueeze(0)
wq2, sc2 = per_group_symmetric_quant(ws, 128, 4)
deq2 = (wq2.float().reshape(k, 2, 128) / sc2.unsqueeze(-1)).reshape(k, n) / s.unsqueeze(0)

err_unscaled = quant_error(w, deq1)
err_scaled   = quant_error(w, deq2)
print(f"整体量化误差: unscaled={err_unscaled:.4f}, AWQ-scaled={err_scaled:.4f}")
if err_scaled > err_unscaled:
    print(f"  ⚠️ AWQ-scaled 的【整体】误差反而更高——这是预期行为，不是 bug：")
    print(f"     AWQ 刻意把缩放预算花在显著 channel（激活幅值大、对输出贡献大的列），")
    print(f"     非显著 channel 的量化误差因此上升。整体均值被这些非显著 channel 拉高，")
    print(f"     但最终损失只由「激活 × 权重」的加权和决定——显著 channel 误差降下来，")
    print(f"     对真实推理输出/perplexity 的影响远大于整体均值的变化。下面逐 channel 看就明白：")

sig_better = []
for ch in [30, 150]:
    eu = (w[:,ch]-deq1[:,ch]).abs().mean()/w[:,ch].abs().mean()
    es = (w[:,ch]-deq2[:,ch]).abs().mean()/w[:,ch].abs().mean()
    print(f"  显著 channel {ch}: unscaled={eu:.4f} -> AWQ-scaled={es:.4f}")
    sig_better.append(es < eu)
# 再看一个非显著 channel，对比"被牺牲"的方向
ch_nonsig = 0
eu0 = (w[:,ch_nonsig]-deq1[:,ch_nonsig]).abs().mean()/w[:,ch_nonsig].abs().mean()
es0 = (w[:,ch_nonsig]-deq2[:,ch_nonsig]).abs().mean()/w[:,ch_nonsig].abs().mean()
print(f"  非显著 channel {ch_nonsig}: unscaled={eu0:.4f} -> AWQ-scaled={es0:.4f}（被牺牲，误差升）")
assert all(sig_better), "AWQ 应降低显著 channel 的量化误差（整体误差可能略升，因缩放改变组分布）"
print("\nL2a PASS：AWQ 缩放显著降低显著 channel 量化误差——这才是 AWQ 的目标（用整体均值的升高换关键 channel 的精度）。")

# tiny Qwen2 gate_proj
from transformers import Qwen2Config, Qwen2ForCausalLM
def make_tiny(vocab=320, hidden=128, inter=512):
    cfg = Qwen2Config(num_hidden_layers=1, hidden_size=hidden, intermediate_size=inter,
        num_attention_heads=4, num_key_value_heads=2, vocab_size=vocab, tie_word_embeddings=True)
    return Qwen2ForCausalLM(cfg).eval()
tiny = make_tiny()
gate_w = tiny.model.layers[0].mlp.gate_proj.weight.data.float()   # [inter, hidden]
a_tiny = torch.randn(32, tiny.config.hidden_size).abs() + 0.01
s_tiny = awq_weight_scale(a_tiny)
# inter 必须能被 128 整除（这里 512 OK）
if gate_w.shape[1] % 128 == 0:
    wq, sc = per_group_symmetric_quant(gate_w.t(), 128, 4)   # [hidden, inter]
    deq = (wq.float().reshape(gate_w.shape[1], gate_w.shape[0]//128, 128) / sc.unsqueeze(-1)).reshape(gate_w.shape[1], gate_w.shape[0]).t()
    err = quant_error(gate_w, deq)
    print(f"\ntiny gate_proj INT4 per-group 量化误差: {err:.4f}")
    assert err < 0.2
    print("L2b PASS：tiny Qwen2 gate_proj per-group INT4 量化跑通")
else:
    print("L2b SKIP：inter 不整除 128（tiny 规模太小）")

## L3：H200 执行（真 0.5B gate_proj，group 64/128/256）

GPU 守卫。在真 0.5B 的一层 gate_proj 上对比不同 group_size 的 INT4 量化误差（group 越小越细、误差越低但 scale 开销越大）。

In [ ]:
if torch.cuda.is_available():
    from transformers import AutoModelForCausalLM, AutoTokenizer
    tok = AutoTokenizer.from_pretrained(TINY_MODEL_DIR)
    model = AutoModelForCausalLM.from_pretrained(TINY_MODEL_DIR, dtype=torch.float16,
                                                 device_map="auto").eval()
    gate_w = model.model.layers[0].mlp.gate_proj.weight.data.float().cpu()  # [inter, hidden]
    W = gate_w.t()   # [hidden, inter]
    inter = W.shape[1]
    print(f"真 0.5B gate_proj: hidden={W.shape[0]}, inter={inter}")
    for gs in [64, 128, 256]:
        if inter % gs != 0:
            print(f"  group_size={gs}: 不整除，跳过"); continue
        wq, sc = per_group_symmetric_quant(W, gs, 4)
        deq = (wq.float().reshape(W.shape[0], inter//gs, gs) / sc.unsqueeze(-1)).reshape(W.shape[0], inter)
        err = quant_error(W, deq)
        print(f"  INT4 group_size={gs}: 量化误差={err:.4f}, 额外 scale 存储={inter//gs*W.shape[0]*4/1e6:.2f}MB")
    del model; torch.cuda.empty_cache()
else:
    print("跳过 L3：无 GPU（CPU 环境只跑 L1/L2）。")

## 产物检查

打印 group_size 对比，写 awq_summary.json。

In [ ]:
def report_awq():
    torch.manual_seed(9)
    k, n = 32, 512
    w = torch.randn(k, n) * 0.08
    a = torch.ones(64, n); a[:, 50] *= 80.0
    s = awq_weight_scale(a)
    summary = {}
    for gs in [64, 128, 256]:
        ws = w * s.unsqueeze(0)
        wq, sc = per_group_symmetric_quant(ws, gs, 4)
        deq = (wq.float().reshape(k, n//gs, gs)/sc.unsqueeze(-1)).reshape(k, n) / s.unsqueeze(0)
        summary[f"group_{gs}"] = {
            "rel_err": quant_error(w, deq),
            "scale_overhead_bytes": int((n//gs)*k*4),
        }
    (OUT_ROOT/"awq_summary.json").write_text(json.dumps(summary, indent=2))
    print("== AWQ W4 group_size 对比 ==")
    for k_, v in summary.items():
        print(f"  {k_}: 相对误差={v['rel_err']:.4f}, scale 开销={v['scale_overhead_bytes']} bytes")
    print("\n结论：group 越小误差越低，但 scale 存储开销上升（W4A16 的精度-开销权衡）。")

report_awq()